# Notebook 1: Synthetic Test-Order Generation

    The objective of this notebook is to generate a realistic and challenging set of synthetic customer order transcripts. We will use these test orders in subsequent notebooks to configure, refine, and benchmark our extraction pipeline.

    While real-world transcripts are ideal for enterprise applications, generating synthetic test data using LLMs provides a low-overhead method to explore LangChain abstractions, model initialization patterns, and schema-enforcement techniques.

### Notebook Workflow
1. Unified Model Initialization with `init_chat_model()`
2. Inspecting the `AIMessage` Object
3. Local Model Execution with `Ollama`
4. Enforcing `Structured Output` using Pydantic
5. `Exercise`: Adversarial Synthetic Data Generation


### Getting Started
Use `uv sync` in the terminal to install dependencies and create the virtual environment for the project. Select your project environment kernel (`pizza-order-extractor (3.13.xx)`)for this notebook and execute the initial setup cell. The `check_apikeys()` helper loads your API keys from the `.env` file into system environment variables for authentication.



In [ ]:
from pizza_order_extractor.helper_functions import check_apikeys

check_apikeys()

## 1. Unified Model Initialization with `init_chat_model()`

LangChain's `init_chat_model()` provides a provider-agnostic interface for initializing chat models. This abstraction allows you to switch between cloud provider APIs (e.g., Anthropic, OpenAI) and local engines (e.g., Ollama) with minimal code modifications.

**Requirements:**
* The provider-specific integration package must be installed in your environment (e.g., `langchain-anthropic`, `langchain-ollama`). Inspect `pyproject.toml` to see available dependencies.
* For additional provider configurations, see the [LangChain Model Initialization Guide](https://docs.langchain.com/oss/python/langchain/models#initialize-a-model) and [Chat Integrations Documentation](https://docs.langchain.com/oss/python/integrations/chat). If necesary, add them to the envionment by using `uv add <package_name>`.

In the next cell, we initialize a lightweight Anthropic model (`claude-haiku-5-5`) and use `.invoke()` to trigger an unconstrained text generation call.

In [ ]:
from langchain.chat_models import init_chat_model

model_name = "claude-haiku-5-5"
provider = "anthropic"

model = init_chat_model(model_name, model_provider=provider)

prompt = """
Generate a pizza order transcribed as if the customer was placing them verbally over the phone. 
Only generate the customer's turn with the order.

"""
response = model.invoke(prompt)
response

## 2. Inspecting the `AIMessage` Object

LangChain returns model outputs wrapped inside an `AIMessage` container. This object contains the text output (`.content`) alongside execution metadata, token usage tracking, and provider details.

Execute the cell below to inspect the internal structure of `AIMessage` using the `print_aimessage()` helper function.

In [ ]:
from pizza_order_extractor.helper_functions import print_aimessage

print_aimessage(response)

### Extracting Raw Content and Token Metrics

To extract specific attributes programmatically (such as message text or token usage), access the corresponding object attributes directly:

In [ ]:
print(response.content)
print()
print(f"Total token usage: {response.usage_metadata['total_tokens']}")

## 3. Local Model Execution with `Ollama`

Next, we will run the same invocation using a locally hosted open-weights model via **Ollama** and the `langchain_ollama` integration package.

**Setup Steps:**
1. Download and install Ollama from [ollama.com](https://ollama.com).
2. Pull an instruct model to your local machine using the terminal:
   ```bash
   ollama pull qwen3:4b-instruct 
   ```
   If you need to start the ollama engine, use `ollama serve` and open a second terminal window to proceed with the downloads. 
   
3. Verify active local models by running `ollama list` in your terminal. 

If you're **not sure which models will run well** on your computer, please ask your chatbot about it.

In [ ]:
model_name = "qwen3:4b-instruct"
provider = "ollama"

model = init_chat_model(model_name, model_provider=provider)

prompt = """
Generate a pizza order transcribed as if the customer was placing them verbally over the phone. 
Only generate the customer's turn with the order.
"""
response = model.invoke(prompt)

print(response.content)
print()
print(f"Total token usage: {response.usage_metadata['total_tokens']}")

## 4. Enforcing `Structured Output` using Pydantic

Unconstrained chat models tend to prepend or append conversational chatter, titles, or markdown formatting (e.g., *"Sure! Here's a natural, verbal pizza order..."*). 

When building automated pipelines, conversational fluff breaks downstream parsing. To ensure deterministic outputs, we define the target data structure using **Pydantic** models and bind them to the LLM via LangChain's `.with_structured_output()` interface.

In [ ]:
# The LLM receives the information enclosed in triple quotation marks ("""), 
# as well as the field description and data type constraints.

from typing import List
from pydantic import BaseModel, Field

class Pizza(BaseModel):
    """Transcription of an individual order turn."""
    pizza_order: str = Field(
        ..., description="Raw transcript of the customer's spoken pizza order."
    )

class PizzaOrderList(BaseModel):
    """Collection of generated synthetic orders."""
    order_list: List[Pizza]

In [ ]:
# the function that binds the schema to the llm. Note that it includes the parameter "temperature". 
# "Temerature" is valid for most local models, but it is deprecated for models like Haiku-5-5. So we defined it as "None".

from langchain.chat_models import init_chat_model

def generate_structured_example_orders(model_name, model_provider, prompt, temperature=None):
    model = init_chat_model(
        model_name,
        model_provider=model_provider,
        temperature=temperature
    )
    structured_llm = model.with_structured_output(schema=PizzaOrderList)
    return structured_llm.invoke(prompt)

In [ ]:
model_name = "qwen3:4b-instruct"
model_provider = "ollama"
temperature = 0.0
prompt = """
Generate 10 pizza orders transcribed as if the customer was placing them verbally over the phone. 
Only generate the customer's turn with the order.
"""

response = generate_structured_example_orders(
    model_name,
    model_provider,
    prompt,
    temperature
)
response

In [ ]:
for order in response.order_list:
    print(order.pizza_order)

In [ ]:
model_name = "claude-haiku-5-5"
model_provider = "anthropic"

prompt = """
Generate 10 pizza orders transcribed as if the customer was placing them verbally over the phone. 
Only generate the customer's turn with the order.
"""

response = generate_structured_example_orders(
    model_name,
    model_provider,
    prompt,
)


for order in response.order_list:
    print(order.pizza_order)

## 5. Exercise: Adversarial Synthetic Data Generation

Now it is your turn to modify the baseline prompt to generate a challenging, edge-case-heavy dataset.

In real-world phone orders, customers speak in a non-linearly and messy way. To build an effective benchmark, apply **adversarial prompt engineering** instructions to force realistic human speech patterns. The minimum requirements should include the following challenges:
* **Different types of pizza in a single order** 
* **Complex “half-and-half” pizzas and custom pizzas** 
* **Corrections or changes to the order**
* **Special intstructions and wishes** 

To archive this, modify the prompt and experiment with the `temperature` parameter (e.g., `0.7` to `1.0`) to increase sample diversity.


Execute generation using your local Ollama model. Once generated, save the dataset using `save_orders_to_file()`. This function exports the structured output into a timestamped JSON file inside the project's `data/` directory.

Please read through the generated orders carefully before saving them. Orders that you do not understand will most likely not be processed correctly by the extractor either. If you want to limit the chaos inside the orders, define a list of allowed pizzas and toppings, just as you would in a real pizzeria.



In [ ]:
model_name = "llama3.1:8b"
model_provider = "ollama"
temperature = 1.0 #Try out different temperatures, and check out how the temperature setting works.
prompt = """
Create 10 pizza orders that are presented as if the customer were placing them verbally over the phone.
Create only the part where the customer places the order.
Here are some guidelines for creating the orders:
The customer can change their mind and modify their selection regarding the pizza details at any time during the ordering process.
The customer can choose from the following pizza options: Pepperoni, Capricciosa, Hawaiian, Four Cheese, Veggie Supreme, BBQ Chicken.
The customer can modify pre-set pizzas by adding or removing toppings.
The customer can order custom pizzas by selecting the toppings themselves.
The customer can choose from the following sauces: tomato sauce, pesto, Buffalo sauce, white sauce, BBQ sauce.
The customer can order up to three pizzas.
The customer can order a split pizza with two different halves.
The customer can also choose the type of pizza crust: thick, regular, or thin.
The customer can specify special requests or provide additional information.
"""
prompt = """
You task is to enerate 10 pizza orders transcribed as if the customer was placing them verbally over the phone. 
Only generate the customers turn with the order.

"""

response = generate_structured_example_orders(
    model_name,
    model_provider,
    prompt,
    temperature
)

for order in response.order_list:
    print(order.pizza_order)

In [ ]:
# When you are happy with the generated sample orders, save them to data/test_orders using this helper_function
from pizza_order_extractor.helper_functions import save_orders_to_file


save_orders_to_file(response, model_name)

## 6. Generating an Evaluation Dataset (Validation Set)

To validate our extractor in Notebooks 2, we require a separate validation dataset.

Re-run the generation pipeline using a cloud model (e.g., Claude Haiku) to produce 10 additional adversarial orders. Keeping an uncollected validation dataset allows us to detect **overfitting**—ensuring prompt improvements in Notebook 2 generalize to unseen order phrasing rather than over-indexing on our primary training batch.

In [ ]:
# Repeat generation using Claude Haiku for the validation set and save the results
model_name = "claude-haiku-5-5"
provider = "anthropic"

validation_response = generate_structured_example_orders(
    model_name,
    provider,
    prompt
)

for order in validation_response.order_list:
    print(order.pizza_order)

save_orders_to_file(validation_response, model_name)